In [1]:
# Program to show how LLM can call a tool based on the requirement
from langchain.chat_models import init_chat_model
from langchain_core.tools import tool
from langchain_core.messages import HumanMessage, ToolMessage, SystemMessage, AIMessage
import requests

In [2]:
@tool
def get_weather(latitude: float, longitude: float) -> dict:
    """function to return weather information based on latitude and longitude"""

    """This is a publically available API that returns the weather for a given location."""
    response = requests.get(
        f"https://api.open-meteo.com/v1/forecast?latitude={latitude}&longitude={longitude}&current_weather=true"
    )
    data = response.json()
    return data["current_weather"]


@tool
def get_coordinates(city_name: str) -> dict:
    """Return latitude & longitude for a city"""
    url = f"https://geocoding-api.open-meteo.com/v1/search?name={city_name}"
    response = requests.get(url)
    data = response.json()
    first = data["results"][0]
    return {"latitude": first["latitude"], "longitude": first["longitude"]}


In [3]:
# gemini-2.5-flash does not have geo capability, so we need to get cooridinates with a tool
gemini = init_chat_model("gemini-2.5-flash", model_provider="google_genai")
gemini_with_tools = gemini.bind_tools([get_coordinates, get_weather])

In [8]:
# gemini-3.5-flash has geo capability 
gemini = init_chat_model("gemini-3.5-flash", model_provider="google_genai")
gemini_with_tools = gemini.bind_tools([get_weather])

In [4]:
system_message = SystemMessage(
    "You are a helpful weather assistant.If needed, find out latitude and logitude then provide to tool. ")

user_message = HumanMessage("What is the weather in Perth")
# user_message = HumanMessage("What is the capital of Australia")

messages = [system_message, user_message]

In [5]:
while True:
    response = gemini_with_tools.invoke(messages)
    messages.append(response)

    if len(response.tool_calls) == 0:
        break

    for tool_call in response.tool_calls:
        func_name = tool_call["name"]
        chosen_function = eval(func_name)

        # Call the tool and add result
        tool_result = chosen_function.invoke(tool_call)
        messages.append(tool_result)    


In [6]:
for message in messages:
    message.pretty_print()

================================ System Message ================================

You are a helpful weather assistant.If needed, find out latitude and logitude then provide to tool. 
================================ Human Message =================================

What is the weather in Perth
================================== Ai Message ==================================
Tool Calls:
  get_coordinates (8c8a1a77-8459-4d94-9aa8-bb81cf59c5b1)
 Call ID: 8c8a1a77-8459-4d94-9aa8-bb81cf59c5b1
  Args:
    city_name: Perth
================================= Tool Message =================================
Name: get_coordinates

{"latitude": -31.95224, "longitude": 115.8614}
================================== Ai Message ==================================
Tool Calls:
  get_weather (e85a60f7-f315-4c04-9262-e8aee608c2f0)
 Call ID: e85a60f7-f315-4c04-9262-e8aee608c2f0
  Args:
    latitude: -31.95224
    longitude: 115.8614
================================= Tool Message =================================